# **Lab 03:** Transfer Learning with PyTorch and Model Improvement

## Graded Individual Activity:

In this activity, we will carry out a series of tests to evaluate the performance of our image classification model. The objective is to correctly identify whether an image contains the face of a specific student or corresponds to a background. To do this, we have defined two main labels:

##### **Label 1:** `Student Name`  
##### **Label 2:** `Background` (images that do not contain the student’s face)

#### **Tests:**

The tests will be conducted using different types of input images to verify the model’s accuracy in scenarios not seen during training:

1. **Input image:** A photo of the student that is not included in the training folder (take a photo on the day of the presentation using your webcam).  
   **Required output label:** `Student Name`

2. **Input image:** A background image that is not included in the training folder (the student presenting before you will give you a search keyword; you may download the image from Google Images).  
   **Required output label:** `Background`

3. **Input image:** A photo of a different face (not the student’s face) that is not included in the training folders (the student presenting before you will give you the name of a celebrity who resembles you; you may download the image from Google Images).  
   **Required output label:** `Background`

##### **Hyperparameter Testing:**

Additionally, you may explore the impact of various hyperparameters on the model’s performance. Examples of hyperparameters to adjust include:

- Increasing the number of images in the dataset  
- Changing the loss function  
- Changing the optimizer  
- Changing the number of training epochs  
- Changing the batch size  

These tests and adjustments will allow you to optimize the model to achieve the best possible accuracy in image classification.

##### **Results Presentation**

Students will present their results in class using a `streamlit` app. The final grade will depend on the successful completion of the 3 tests.

In [15]:
import os
from PIL import Image
from torchvision import transforms

input_folder = "/home/mateo/Documentos/Computer_Vision/UC.03 Transfer Learning & Model Improvement/dataset/mateo/original"
output_folder = "/home/mateo/Documentos/Computer_Vision/UC.03 Transfer Learning & Model Improvement/dataset/mateo/augmented"

os.makedirs(output_folder, exist_ok=True)

transform = transforms.Compose([
    transforms.RandomRotation(30),
    transforms.RandomHorizontalFlip(p=1),
    transforms.ColorJitter(brightness=0.3),
])

for img_name in os.listdir(input_folder):

    img_path = os.path.join(input_folder, img_name)
    image = Image.open(img_path).convert("RGB")

    for i in range(6):

        aug_img = transform(image)

        new_name = f"{img_name.split('.')[0]}_aug_{i}.jpg"
        aug_img.save(os.path.join(output_folder, new_name))

In [2]:
import os
from PIL import Image
from torchvision import transforms

input_folder = "/home/mateo/Documentos/Computer_Vision/UC.03 Transfer Learning & Model Improvement/dataset/background/original"
output_folder = "/home/mateo/Documentos/Computer_Vision/UC.03 Transfer Learning & Model Improvement/dataset/background/augmented"

os.makedirs(output_folder, exist_ok=True)

transform = transforms.Compose([
    transforms.RandomRotation(20),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
])

for img_name in os.listdir(input_folder):

    img_path = os.path.join(input_folder, img_name)
    image = Image.open(img_path).convert("RGB")

    for i in range(4):

        aug_img = transform(image)

        new_name = f"{img_name.split('.')[0]}_aug_{i}.jpg"
        aug_img.save(os.path.join(output_folder, new_name))

In [18]:
import torch.optim as optim

def build_model(num_classes=2):
    model = models.resnet18(weights='IMAGENET1K_V1')
    
    # Congelar capas iniciales para Transfer Learning
    for param in model.parameters():
        param.requires_grad = False
        
    # Cambiar la última capa (Fully Connected) para tus 2 etiquetas: Mateo y Background
    num_ftrs = model.fc.in_features
    model.fc = nn.Linear(num_ftrs, num_classes)
    
    return model

model = build_model(num_classes=2)
criterion = nn.CrossEntropyLoss() # Recomendado en la rúbrica cambiar funciones de pérdida
optimizer = optim.Adam(model.fc.parameters(), lr=0.001) # Optimizer sugerido

In [3]:
from torchvision import transforms

transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])  
])

In [5]:
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, random_split

dataset = ImageFolder(
    "/home/mateo/Documentos/Computer_Vision/UC.03 Transfer Learning & Model Improvement/dataset_final",
    transform=transform
)

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)

In [6]:
import torch
import torch.nn as nn
from torchvision import models

model = models.resnet18(pretrained=True)

for param in model.parameters():
    param.requires_grad = False

num_features = model.fc.in_features

model.fc = nn.Linear(num_features, 2)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = model.to(device)

/home/mateo/miniconda3/envs/vision_computacional/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/mateo/miniconda3/envs/vision_computacional/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /home/mateo/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100.0%


In [7]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(model.fc.parameters(), lr=0.001)

In [8]:
epochs = 5

for epoch in range(epochs):

    model.train()
    running_loss = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {running_loss/len(train_loader)}")

Epoch 1, Loss: 0.3810817130974361
Epoch 2, Loss: 0.17330264087234223
Epoch 3, Loss: 0.11684099691254753
Epoch 4, Loss: 0.08353501678045307
Epoch 5, Loss: 0.07480084550167833


In [9]:
correct = 0
total = 0

model.eval()

with torch.no_grad():

    for images, labels in val_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)

        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total

print("Accuracy:", accuracy)

Accuracy: 99.5475113122172


In [10]:
torch.save(model.state_dict(), "mateo_classifier.pth")

In [11]:
import torch
import torch.nn as nn
from torchvision import models

model = models.resnet18(pretrained=False)

num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 2)

model.load_state_dict(torch.load("mateo_classifier.pth"))

model.eval()

/home/mateo/miniconda3/envs/vision_computacional/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [12]:
from torchvision import transforms

transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
])

In [13]:
from PIL import Image

classes = ["background", "mateo"]

def predict_image(image_path):

    image = Image.open(image_path).convert("RGB")

    image = transform(image)

    image = image.unsqueeze(0)

    with torch.no_grad():
        outputs = model(image)
        _, predicted = torch.max(outputs, 1)

    return classes[predicted.item()]

In [14]:
image_path = "/home/mateo/Imágenes/test.jpg"

prediction = predict_image(image_path)

print("Prediction:", prediction)

Prediction: background


### **Grading Rubric (10 points)**

The Streamlit app should be working correctly during presentation to get points.

| Criterion | Points |
|----------|--------|
| Test 1: Correct classification of student's new photo (`Student Name`) | 2.5 |
| Test 2: Correct classification of unseen background image (`Background`) | 2.5 |
| Test 3: Correct classification of a different face (celebrity look-alike) as `Background` | 5 |
| **Total** | **10** |

### **Scoring Rules**

- Each test is graded as **pass (full points)** or **fail (0 points)**.
- If the model predicts the required label correctly → full points for that test.
- If the prediction is incorrect → 0 points for that test.
- Test 3 has the highest weight because it evaluates the model’s ability to generalize and avoid false positives.

---

<p style="text-align: right; font-size:14px; color:gray;">
<b>Prepared by:</b><br>
Manuel Eugenio Morocho-Cayamcela
</p>